# HeadMouse: Colab session viewer
Run the next two cells. The single head/cursor viewer appears above the import controls and participant table.
Use a CPU runtime; no camera, mouse driver, desktop GUI or GPU is needed.

**Privacy:** recordings uploaded here go to your Google Colab runtime, not to GitHub or any playback service. Do not share a notebook with saved outputs containing participant data. Runtime files and imports are temporary; download the summary CSV before disconnecting.

**Setup:** the repository/branch below must contain `colab_session_viewer`. If working with unpublished changes, upload the complete project folder to the runtime first and set `PROJECT_DIR` to it.

In [ ]:
from pathlib import Path
import subprocess
import sys
import urllib.request

REPOSITORY_URL = 'https://github.com/simonjudge/HeadMouse.git'
REVISION = ''  # Optional published branch or tag; empty uses the default branch.
PROJECT_DIR = Path('/content/HeadMouse')

if not PROJECT_DIR.exists():
    command = ['git', 'clone', '--depth', '1']
    if REVISION:
        command += ['--branch', REVISION]
    subprocess.check_call(command + [REPOSITORY_URL, str(PROJECT_DIR)])
requirements = PROJECT_DIR / 'colab_session_viewer' / 'requirements.txt'
if not requirements.is_file():
    raise FileNotFoundError('This checkout does not contain the Colab viewer. Select its published branch or upload the complete changed project.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from session_viewer.mesh_visualizer import OBJ_PATH, MEDIAPIPE_OBJ_URL
if not OBJ_PATH.exists():
    OBJ_PATH.parent.mkdir(parents=True, exist_ok=True)
    # Use Python's default HTTPS certificate verification.
    with urllib.request.urlopen(MEDIAPIPE_OBJ_URL, timeout=60) as response:
        mesh = response.read()
    OBJ_PATH.write_bytes(mesh)
print('Ready. Run the viewer cell below.')


In [ ]:
from google.colab import output
output.enable_custom_widget_manager()
from colab_session_viewer.app import NotebookViewer

viewer = NotebookViewer()
viewer.show();


## Importing multiple participants and sessions
For each test, enter a participant (for example `P01`) and session (`Test 1`) and choose a head JSON/CSV, a Fitts JSON, or both. Click **Add session**, then repeat with `P01 / Test 2`, `P02 / Test 1`, etc. Choose both files only when recorded together with overlapping host timestamps. Legacy files without host timestamps can be imported separately; they are never silently aligned.

To upload several data sets at once, put a `manifest.csv` at the root of a ZIP with columns:
```csv
participant,session,head_file,fitts_file
P01,Test 1,P01/head1.json,P01/fitts1.json
P01,Test 2,P01/head2.csv,P01/fitts2.json
P02,Test 1,,P02/fitts1.json
```
Paths are relative to the ZIP, use `/` separators, and may be blank when that stream was not recorded. Use **Dataset ZIP**, then **Import dataset ZIP**. You can import multiple ZIPs without clearing earlier data. Duplicate participant/session names are rejected rather than overwritten.

The per-participant exclusion control removes the first **N targets per Fitts file**, not entire tests and not repetitions of each condition. The single summary row pools all remaining targets for that participant using the same FittsStats formulas as the desktop viewer. Playback, head measurements and pre-test summaries retain all data. If N exhausts a file, that file contributes zero pointing targets; unavailable measures are blank.

The viewer has play/pause, rewind, seek, 0.1–5x speed, graph zoom, a rotatable head mesh, live values, target/click markers and pre-test playback. The browser renderer replaces Qt/OpenGL; recording times, neutral pose, coordinate transforms, angular dynamics and summary formulas follow the desktop viewer.

For recordings already on Google Drive, mount Drive in a separate cell and call:
```python
viewer.add_session('P03', 'Test 1',
    head_path='/content/drive/MyDrive/study/head.json',
    fitts_path='/content/drive/MyDrive/study/fitts.json')
```
Only run the viewer cell once per collection: rerunning it starts a new empty collection.